# Distributed Queries with Spark SQL — RetailMax Sales Analytics
## Branch and Category Performance Analysis with PySpark SQL

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 2–5 — Data Understanding · Preparation · Modeling · Evaluation |
| **Module** | 9 — Big Data & Spark (Alkemy Bootcamp) |
| **Dataset** | RetailMax Sales · synthetic transactional data |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This case applies Spark SQL to perform distributed analytical queries over RetailMax
> sales transactions. Three queries demonstrate filtering, aggregation, and ordering
> on a registered temporary view, with the Catalyst optimizer explaining how Spark
> rewrites and accelerates execution. Results inform inventory allocation, marketing
> budget assignment, and branch performance benchmarking decisions.

## Table of Contents

1. [Environment Setup & Data Loading](#1-environment-setup--data-loading)
2. [Data Exploration](#2-data-exploration)
3. [Temp View Registration](#3-temp-view-registration)
4. [SQL Queries — Filter, Aggregation, Ordering](#4-sql-queries--filter-aggregation-ordering)
5. [Catalyst Optimizer — Execution Plan](#5-catalyst-optimizer--execution-plan)
6. [Business Synthesis](#6-business-synthesis)
7. [LEAN Filter — Waste Elimination Review](#7-lean-filter--waste-elimination-review)
8. [Decisions Log](#8-decisions-log)
9. [Next Steps](#9-next-steps)

---
## 1. Environment Setup & Data Loading
### CRISP-DM Phase 2 — Data Understanding

In [ ]:
# ===== Environment Setup =====
import os
import sys
import warnings
from pathlib import Path

# Force Spark workers to use the same Python as this notebook
# Must be set BEFORE importing PySpark
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

# ===== Paths =====
_CASE_ROOT = Path(os.getcwd()).parent  # notebooks/ → case root
DATA_PATH  = _CASE_ROOT / "data" / "raw" / "ventas.csv"
assert DATA_PATH.exists(), "ventas.csv not found in data/raw/"

# ===== PySpark imports =====
from pyspark.sql import SparkSession

# ===== Spark Session =====
spark = (
    SparkSession.builder
    .appName("Case - Spark SQL Queries")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")

In [ ]:
# ===== Data Loading =====
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_PATH))
)

print(f"Records loaded : {df_raw.count():,}")
print(f"Columns        : {len(df_raw.columns)}")
df_raw.printSchema()
df_raw.show(5, truncate=False)

---
## 2. Data Exploration
### CRISP-DM Phase 2 — Data Understanding

**Lean filter:** explore only what informs the business decision —
distribution by category, branch, and revenue range.

In [ ]:
# ===== Numeric summary =====
print("=== Numeric summary — monto ===")
df_raw.select("monto").describe().show()

# ===== Distinct categories and branches =====
print("=== Distinct values ===")
print(f"Categories : {df_raw.select('categoria').distinct().count()}")
print(f"Branches   : {df_raw.select('id_sucursal').distinct().count()}")
print(f"Products   : {df_raw.select('producto').distinct().count()}")

---
## 3. Temp View Registration
### CRISP-DM Phase 3 — Data Preparation

Register the DataFrame as a SQL temporary view to enable distributed SQL queries
through the Catalyst engine.

In [ ]:
# ===== Register temp view =====
df_raw.createOrReplaceTempView("ventas")
print("Temporary view 'ventas' registered.")

# Verify
spark.sql("SELECT COUNT(*) AS total_records FROM ventas").show()

---
## 4. SQL Queries — Filter, Aggregation, Ordering
### CRISP-DM Phase 4 — Modeling (analytical queries)

Three queries demonstrating the core Spark SQL operations required by the consigna:
filters (`WHERE`), aggregations (`GROUP BY` + `SUM`/`COUNT`/`AVG`), and ordering (`ORDER BY`).

### Query 1 — Filter
**Business question:** Which high-value Technology transactions exceed CLP 300,000?
This identifies premium product sales that justify priority restocking.

In [ ]:
q1 = spark.sql("""
    SELECT
        id_venta,
        id_sucursal,
        producto,
        monto,
        fecha
    FROM ventas
    WHERE categoria = 'Tecnologia'
      AND monto > 300000
    ORDER BY monto DESC
""")

q1.show(10, truncate=False)
print(f"High-value Technology transactions: {q1.count()}")

### Query 2 — Aggregation by Category
**Business question:** Which category generates the highest revenue, and what is
the average ticket per category? This informs marketing budget allocation.

In [ ]:
q2 = spark.sql("""
    SELECT
        categoria,
        COUNT(*)                   AS total_ventas,
        SUM(monto)                 AS ingreso_total,
        ROUND(AVG(monto), 0)       AS ticket_promedio
    FROM ventas
    GROUP BY categoria
    ORDER BY ingreso_total DESC
""")

q2.show()

### Query 3 — Top 5 Branches by Revenue
**Business question:** Which branches lead in total revenue? Top performers serve
as benchmarks for operational best practices.

In [ ]:
q3 = spark.sql("""
    SELECT
        id_sucursal,
        COUNT(*)              AS total_ventas,
        SUM(monto)            AS ingreso_total,
        ROUND(AVG(monto), 0)  AS ticket_promedio
    FROM ventas
    GROUP BY id_sucursal
    ORDER BY ingreso_total DESC
    LIMIT 5
""")

q3.show()

---
## 5. Catalyst Optimizer — Execution Plan
### CRISP-DM Phase 5 — Evaluation

Catalyst is Spark's query optimizer. It rewrites SQL into an efficient physical
execution plan through four stages:

1. **Parsed Logical Plan** — Parses SQL into an unresolved tree.
2. **Analyzed Logical Plan** — Resolves column names and types against the catalog.
3. **Optimized Logical Plan** — Applies rule-based optimizations:
   - *Predicate pushdown* — pushes `WHERE` filters as close to the data source as possible.
   - *Projection pruning* — removes columns not needed downstream.
   - *Constant folding* — evaluates constant expressions at compile time.
4. **Physical Plan** — Selects execution strategies (hash aggregation, sort, broadcast).

The `explain(True)` method exposes all four stages.

In [ ]:
# ===== Catalyst execution plan for Q1 =====
print("=" * 60)
print("  EXECUTION PLAN — Q1 (filter + ordering)")
print("=" * 60)
q1.explain(True)

---
## 6. Business Synthesis
### CRISP-DM Phase 5 — Evaluation

In [ ]:
print("=" * 55)
print("  EXECUTIVE SUMMARY — RetailMax Sales SQL Queries")
print("=" * 55)

print(f"""
DATASET
  Total records       : {df_raw.count():,}
  Categories          : {df_raw.select('categoria').distinct().count()}
  Branches            : {df_raw.select('id_sucursal').distinct().count()}

QUERY RESULTS
  Q1 — High-value Tech transactions (>CLP 300K) : {q1.count()}
  Q2 — Revenue ranking by category               : computed
  Q3 — Top 5 branches by revenue                 : computed

BUSINESS INTERPRETATION:
  Spark SQL enables distributed analytical queries over the full
  transactional history without loading data into a single machine.
  The Catalyst optimizer pushes filters down and prunes projections
  automatically — the same query scales from thousands to billions
  of records with no code changes.

  These three query patterns (filter, aggregation, top-N) cover
  approximately 80% of operational analytics requests in retail.
""")

---
## 7. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Does every query link to a business decision? | ✅ | Q1 → restocking; Q2 → marketing budget; Q3 → branch benchmarking |
| Is there redundancy between queries? | ✅ | Each query answers a distinct question — no duplication |
| Is there a simpler way to answer the question? | ✅ | SQL is the simplest declarative form for these patterns — no procedural code added |
| Were non-informative columns explicitly excluded? | ✅ | Q2 and Q3 select only the columns needed for the business answer |

---
## 8. Decisions Log

| # | Decision | Rationale | LEAN Value |
|---|---|---|---|
| 1 | Use `inferSchema=True` instead of explicit schema | Dataset is small and exploratory — schema inference adds no measurable cost | Simplicity |
| 2 | Register temp view instead of permanent table | Case scope is single-session analysis — no need for catalog persistence | No over-engineering |
| 3 | Three queries (filter, aggregation, top-N) | Covers the three core SQL patterns required by the consigna and by ~80% of retail analytics | 80/20 principle |
| 4 | `explain(True)` only on Q1 | Filter + ordering plan is the most illustrative for Catalyst optimizations — other queries follow the same logic | Avoid redundancy |
| 5 | Resolve `DATA_PATH` from environment variable with fallback | Keeps personal absolute paths out of the public notebook | Privacy / portability |

---
## 9. Next Steps

- [ ] Convert `ventas.csv` to Parquet for faster repeated queries
- [ ] Add window functions (`ROW_NUMBER`, `RANK`) for top-N per branch
- [ ] Add temporal queries (`MONTH(fecha)`, `YEAR(fecha)`) for seasonality analysis
- [ ] Migrate to AWS EMR Serverless + S3 for production-scale execution
- [ ] Close SparkSession

---

**Case:** `case-spark-sql-queries`

In [ ]:
# ===== Close SparkSession =====
spark.stop()
print("SparkSession closed.")